# 02 — Multi-Horizon Forecasting & Peak-Weighted Evaluation

**Objectives:**
1. Evaluate forecast accuracy at horizons h+1 through h+6
2. Introduce peak-weighted MAPE as the primary metric for BESS applications
3. Assess mock forecast temperature (Tx leads) as a feature

**Dataset:** North grid, 2017–2022  
**Models:** Random Forest, LightGBM (Linear Regression dropped — behaviour well understood from 01)  
**Test set:** Last 365 days (2021-07-02 → 2022-07-01)

## 0. Setup

In [ ]:
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/mcyc/predictive-ml.git'])

import sys
sys.path.append('/kaggle/working/predictive-ml/load-forecast/src/')

In [ ]:
from data import load_data, split_chronological
from features import build_features, FEATURES, TARGET, add_tx_leads
from evaluate import (
    evaluate, peak_weighted_evaluate, compare_models,
    build_results_df, plot_residuals, plot_residuals_by_time
)
from models import train_rf, train_lgbm, get_feature_importance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('All imports successful.')

In [ ]:
DATA_PATH       = '/kaggle/input/datasets/mcycee/northern-taiwan-power-data/power_north_taiwan.parquet'  # <-- update
TEST_DAYS       = 365
HORIZONS        = [1, 2, 3, 4, 5, 6]  # hours ahead
PEAK_THRESHOLD  = 0.90                 # top 10% of load = peak

## 1. Load & Build Features

In [ ]:
df_raw = load_data(DATA_PATH)
df     = build_features(df_raw)
print(df.shape)
df.head()

## 2. Baseline: h+1 with Full Feature Set

Reproduce the h+1 result from notebook 01 as a sanity check before extending horizons.

In [ ]:
X_train, X_test, y_train, y_test = split_chronological(
    df, features=FEATURES, target=TARGET, test_days=TEST_DAYS
)

rf_h1   = train_rf(X_train, y_train)
lgbm_h1 = train_lgbm(X_train, y_train, X_test, y_test)

results_h1 = []
evaluate('RF h+1',   y_test, rf_h1.predict(X_test),   results=results_h1)
evaluate('LGBM h+1', y_test, lgbm_h1.predict(X_test), results=results_h1)
compare_models(results_h1)

In [ ]:
# Peak-weighted baseline
peak_results_h1 = []
peak_weighted_evaluate('RF h+1',   y_test, rf_h1.predict(X_test),
                       threshold=PEAK_THRESHOLD, results=peak_results_h1)
peak_weighted_evaluate('LGBM h+1', y_test, lgbm_h1.predict(X_test),
                       threshold=PEAK_THRESHOLD, results=peak_results_h1)
compare_models(peak_results_h1, peak=True)

## 3. Multi-Horizon Experiment

For each horizon h, the target is `north_clean` shifted back by h hours.
Lag features are adjusted so that `lag_1h` becomes the load at t-(h),
i.e. the most recent observation available at forecast time.

**Note:** For h > 1, `lag_1h` refers to the load h hours ago, not 1 hour ago.
We retrain separate models per horizon rather than chaining predictions,
which gives a cleaner upper bound on achievable accuracy at each horizon.

**Leakage prevention:** For horizon h, any lag feature with a period < h is
dropped from the feature set, since those observations would not be available
at forecast time. Concretely:
- h+1 → all lags available (lag_1h, lag_24h, lag_168h)
- h+2 → lag_1h dropped; lag_24h, lag_168h kept
- h+3 through h+6 → same rule applied progressively

In [ ]:
def get_features_for_horizon(base_features, horizon):
    """
    Filter the feature list to remove lag columns whose period is shorter
    than `horizon`, preventing data leakage.

    Lag columns are identified by the pattern 'lag_{N}h' where N is an integer.
    A lag of N hours is only available at forecast horizon h if N >= h.

    Parameters
    ----------
    base_features : list[str]
        Full feature list as defined in features.py (FEATURES constant).
    horizon : int
        Forecast horizon in hours.

    Returns
    -------
    list[str]
        Feature list safe to use at the given horizon.
    """
    import re
    valid = []
    for col in base_features:
        m = re.match(r'lag_(\d+)h', col)
        if m:
            lag_period = int(m.group(1))
            if lag_period >= horizon:      # only keep lags that are available
                valid.append(col)
        else:
            valid.append(col)              # non-lag feature — always keep
    return valid


def build_horizon_dataset(df, horizon, base_features, target, test_days):
    """
    Build train/test splits for a given forecast horizon.

    The target is shifted forward by `horizon` rows (each row = 1 hour after
    resampling), so row t predicts load at t+horizon. Rows with NaN in either
    the target or the feature columns are dropped before splitting.

    Parameters
    ----------
    df : pd.DataFrame
        Feature-engineered dataframe (output of build_features).
    horizon : int
        Forecast horizon in hours.
    base_features : list[str]
        Full feature list; will be filtered for leakage.
    target : str
        Name of the target column in df.
    test_days : int
        Number of days to hold out as test set (chronological tail).

    Returns
    -------
    tuple : (X_train, X_test, y_train, y_test, features_h)
    """
    features_h = get_features_for_horizon(base_features, horizon)

    df_h = df.copy()
    # Shift target: row t carries the load h hours in the future
    df_h[target] = df_h[target].shift(-horizon)

    # Drop rows where either target or any required feature is NaN
    cols_needed = features_h + [target]
    df_h = df_h[cols_needed].dropna()

    X_train, X_test, y_train, y_test = split_chronological(
        df_h, features=features_h, target=target, test_days=test_days
    )
    return X_train, X_test, y_train, y_test, features_h

In [ ]:
# ---------------------------------------------------------------------------
# Multi-horizon loop
# Stores per-horizon results in two flat lists:
#   horizon_results      — global MAPE / RMSE / MAE
#   horizon_peak_results — peak-weighted MAPE (top 10%)
# Also caches trained models and test sets for later residual inspection.
# ---------------------------------------------------------------------------

horizon_results      = []   # list of dicts from evaluate()
horizon_peak_results = []   # list of dicts from peak_weighted_evaluate()
models_cache         = {}   # {(model_name, horizon): fitted model}
test_cache           = {}   # {horizon: (X_test, y_test)}
features_cache       = {}   # {horizon: feature_list}

for h in HORIZONS:
    print(f"\n{'='*60}")
    print(f"  Horizon h+{h}")
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test, features_h = build_horizon_dataset(
        df, horizon=h, base_features=FEATURES,
        target=TARGET, test_days=TEST_DAYS
    )

    print(f"  Features ({len(features_h)}): {features_h}")
    print(f"  Train rows: {len(X_train):,}   Test rows: {len(X_test):,}")

    # --- Train ---
    rf_h   = train_rf(X_train, y_train)
    lgbm_h = train_lgbm(X_train, y_train, X_test, y_test)

    # --- Global metrics ---
    evaluate(f'RF h+{h}',   y_test, rf_h.predict(X_test),   results=horizon_results)
    evaluate(f'LGBM h+{h}', y_test, lgbm_h.predict(X_test), results=horizon_results)

    # --- Peak-weighted metrics ---
    peak_weighted_evaluate(
        f'RF h+{h}',   y_test, rf_h.predict(X_test),
        threshold=PEAK_THRESHOLD, results=horizon_peak_results
    )
    peak_weighted_evaluate(
        f'LGBM h+{h}', y_test, lgbm_h.predict(X_test),
        threshold=PEAK_THRESHOLD, results=horizon_peak_results
    )

    # --- Cache ---
    models_cache[('RF',   h)] = rf_h
    models_cache[('LGBM', h)] = lgbm_h
    test_cache[h]             = (X_test, y_test)
    features_cache[h]         = features_h

print("\nAll horizons complete.")

In [ ]:
# Tabular summary — global metrics
print("\n=== Global metrics across horizons ===")
compare_models(horizon_results)

In [ ]:
# Tabular summary — peak-weighted metrics
print("\n=== Peak-weighted metrics across horizons ===")
compare_models(horizon_peak_results, peak=True)

## 4. Mock Forecast Temperature (Tx Leads)

Add Tx lead features as a proxy for weather forecast temperature.
Compare h+1 accuracy with and without Tx_lead_{h}h to assess
whether forecast temperature is worth pursuing in production.

**Methodology:** For each horizon h, we add `Tx_lead_{h}h` (the actual future
temperature, known in hindsight) as an oracle upper-bound on what a weather
forecast could provide. The delta MAPE vs the no-lead baseline quantifies
the maximum achievable improvement from a perfect temperature forecast.

In [ ]:
# ---------------------------------------------------------------------------
# Rebuild features with Tx leads (drop_na=False so add_tx_leads can shift)
# add_tx_leads() is expected to add columns 'Tx_lead_1h' … 'Tx_lead_6h'
# ---------------------------------------------------------------------------

df_raw_tx = load_data(DATA_PATH)
df_tx     = build_features(df_raw_tx, drop_na=False)  # keep NaN rows for shifting
df_tx     = add_tx_leads(df_tx, horizons=HORIZONS)    # adds Tx_lead_{h}h columns

print(f"Tx lead columns added: {[c for c in df_tx.columns if 'Tx_lead' in c]}")
print(f"Shape before final dropna: {df_tx.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Tx leads experiment loop
#
# For each horizon h:
#   - Baseline feature set: horizon-safe FEATURES (no Tx lead)
#   - Augmented feature set: baseline + Tx_lead_{h}h
#   - Train RF (faster iteration) and LightGBM on both
#   - Record delta MAPE = MAPE_baseline - MAPE_with_lead (positive = improvement)
# ---------------------------------------------------------------------------

tx_results   = []   # flat list for compare_models
delta_records = []  # {horizon, model, delta_mape, delta_peak_mape}

for h in HORIZONS:
    print(f"\n--- Tx leads | Horizon h+{h} ---")

    tx_lead_col = f'Tx_lead_{h}h'

    # Build base feature set for this horizon (leakage-safe)
    features_base = get_features_for_horizon(FEATURES, h)
    features_aug  = features_base + [tx_lead_col]      # add oracle lead

    # --- Shared target: shift by h on the Tx-lead dataframe ---
    df_h_tx = df_tx.copy()
    df_h_tx[TARGET] = df_h_tx[TARGET].shift(-h)

    def make_split(feat_list):
        """Drop NaN for a given feature list and split chronologically."""
        cols = feat_list + [TARGET]
        df_clean = df_h_tx[cols].dropna()
        return split_chronological(
            df_clean, features=feat_list,
            target=TARGET, test_days=TEST_DAYS
        )

    # Baseline split & train
    Xtr_b, Xte_b, ytr_b, yte_b = make_split(features_base)
    rf_base   = train_rf(Xtr_b, ytr_b)
    lgbm_base = train_lgbm(Xtr_b, ytr_b, Xte_b, yte_b)

    r_rf_base   = []; evaluate(f'RF base h+{h}',   yte_b, rf_base.predict(Xte_b),   results=r_rf_base)
    r_lgbm_base = []; evaluate(f'LGBM base h+{h}', yte_b, lgbm_base.predict(Xte_b), results=r_lgbm_base)
    pr_rf_base   = []; peak_weighted_evaluate(f'RF base h+{h}',   yte_b, rf_base.predict(Xte_b),   threshold=PEAK_THRESHOLD, results=pr_rf_base)
    pr_lgbm_base = []; peak_weighted_evaluate(f'LGBM base h+{h}', yte_b, lgbm_base.predict(Xte_b), threshold=PEAK_THRESHOLD, results=pr_lgbm_base)

    # Augmented split & train
    Xtr_a, Xte_a, ytr_a, yte_a = make_split(features_aug)
    rf_aug   = train_rf(Xtr_a, ytr_a)
    lgbm_aug = train_lgbm(Xtr_a, ytr_a, Xte_a, yte_a)

    r_rf_aug   = []; evaluate(f'RF +Tx h+{h}',   yte_a, rf_aug.predict(Xte_a),   results=r_rf_aug)
    r_lgbm_aug = []; evaluate(f'LGBM +Tx h+{h}', yte_a, lgbm_aug.predict(Xte_a), results=r_lgbm_aug)
    pr_rf_aug   = []; peak_weighted_evaluate(f'RF +Tx h+{h}',   yte_a, rf_aug.predict(Xte_a),   threshold=PEAK_THRESHOLD, results=pr_rf_aug)
    pr_lgbm_aug = []; peak_weighted_evaluate(f'LGBM +Tx h+{h}', yte_a, lgbm_aug.predict(Xte_a), threshold=PEAK_THRESHOLD, results=pr_lgbm_aug)

    # Accumulate for later compare_models display
    tx_results.extend(r_rf_base + r_lgbm_base + r_rf_aug + r_lgbm_aug)

    # Delta MAPE records (positive = Tx lead helps)
    for model_name, r_base, r_aug, pr_base, pr_aug in [
        ('RF',   r_rf_base[0],   r_rf_aug[0],   pr_rf_base[0],   pr_rf_aug[0]),
        ('LGBM', r_lgbm_base[0], r_lgbm_aug[0], pr_lgbm_base[0], pr_lgbm_aug[0]),
    ]:
        delta_records.append({
            'horizon'        : h,
            'model'          : model_name,
            'mape_base'      : r_base['MAPE'],
            'mape_aug'       : r_aug['MAPE'],
            'delta_mape'     : r_base['MAPE']   - r_aug['MAPE'],     # positive = improvement
            'peak_mape_base' : pr_base['peak_MAPE'],
            'peak_mape_aug'  : pr_aug['peak_MAPE'],
            'delta_peak_mape': pr_base['peak_MAPE'] - pr_aug['peak_MAPE'],
        })
        print(f"  {model_name:4s}  MAPE: {r_base['MAPE']:.3f}% → {r_aug['MAPE']:.3f}%  "
              f"(Δ {r_base['MAPE']-r_aug['MAPE']:+.3f}%)  "
              f"Peak-MAPE: {pr_base['peak_MAPE']:.3f}% → {pr_aug['peak_MAPE']:.3f}%  "
              f"(Δ {pr_base['peak_MAPE']-pr_aug['peak_MAPE']:+.3f}%)")

delta_df = pd.DataFrame(delta_records)
print("\nTx leads delta summary:")
print(delta_df.to_string(index=False))

## 5. Results Summary

In [ ]:
# ---------------------------------------------------------------------------
# Helper: extract per-horizon metric tables from the flat result lists
# ---------------------------------------------------------------------------

import re

def results_to_df(result_list, metric_key='MAPE'):
    """
    Pivot a flat list of evaluate() / peak_weighted_evaluate() dicts into
    a (horizon × model) DataFrame for plotting.

    Each dict is expected to have a 'name' field like 'RF h+3' or 'LGBM h+5'.
    """
    rows = []
    for r in result_list:
        m = re.match(r'(RF|LGBM) h\+(\d+)', r['name'])
        if m:
            rows.append({
                'model'  : m.group(1),
                'horizon': int(m.group(2)),
                'value'  : r[metric_key],
            })
    return pd.DataFrame(rows).pivot(index='horizon', columns='model', values='value')

In [ ]:
# ---------------------------------------------------------------------------
# Figure 1: Global MAPE and Peak-MAPE vs horizon — RF vs LightGBM
# ---------------------------------------------------------------------------

mape_df      = results_to_df(horizon_results,      metric_key='MAPE')
peak_mape_df = results_to_df(horizon_peak_results, metric_key='peak_MAPE')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Multi-Horizon Forecast Accuracy — North Grid', fontsize=14, y=1.01)

colors = {'RF': '#2196F3', 'LGBM': '#FF5722'}
markers = {'RF': 'o', 'LGBM': 's'}

for ax, df_plot, ylabel, title in [
    (axes[0], mape_df,      'MAPE (%)',      'Global MAPE vs Horizon'),
    (axes[1], peak_mape_df, 'Peak-MAPE (%)', 'Peak-Weighted MAPE vs Horizon (top 10%)'),
]:
    for model in df_plot.columns:
        ax.plot(
            df_plot.index, df_plot[model],
            label=model, color=colors[model], marker=markers[model],
            linewidth=2, markersize=7
        )
    ax.set_xlabel('Horizon (hours ahead)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(HORIZONS)
    ax.legend()
    ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('horizon_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: horizon_accuracy.png")

In [ ]:
# ---------------------------------------------------------------------------
# Figure 2: Delta MAPE from Tx leads — how much does a perfect temperature
# forecast help at each horizon?
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Improvement from Oracle Tx Lead (perfect temp forecast)',
             fontsize=14, y=1.01)

for ax, ycol, ylabel, title in [
    (axes[0], 'delta_mape',      'ΔMAPE (pp)',      'Global MAPE reduction from Tx lead'),
    (axes[1], 'delta_peak_mape', 'ΔPeak-MAPE (pp)', 'Peak-MAPE reduction from Tx lead'),
]:
    for model, grp in delta_df.groupby('model'):
        ax.plot(
            grp['horizon'], grp[ycol],
            label=model, color=colors[model], marker=markers[model],
            linewidth=2, markersize=7
        )
    ax.axhline(0, color='grey', linewidth=1, linestyle='--', alpha=0.6)
    ax.set_xlabel('Horizon (hours ahead)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(HORIZONS)
    ax.legend()
    ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('tx_lead_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: tx_lead_delta.png")

In [ ]:
# ---------------------------------------------------------------------------
# Figure 3: Combined 4-panel — MAPE, Peak-MAPE, with and without Tx leads
# for the best model (LightGBM) for a clean presentation slide
# ---------------------------------------------------------------------------

# Extract LGBM base vs augmented from tx_results
def extract_tx_series(tx_results, model_prefix, metric_key='MAPE'):
    """Return {horizon: metric} for base and +Tx variants."""
    base, aug = {}, {}
    for r in tx_results:
        m = re.match(rf'({model_prefix}) (base|\+Tx) h\+(\d+)', r['name'])
        if m:
            tag = m.group(2)
            h   = int(m.group(3))
            if tag == 'base':
                base[h] = r[metric_key]
            else:
                aug[h] = r[metric_key]
    return base, aug


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('LightGBM: effect of oracle Tx lead on global and peak MAPE',
             fontsize=13, y=1.01)

for ax, metric_key, peak_key, ylabel, title in [
    (axes[0], 'MAPE',      None,        'MAPE (%)',      'Global MAPE'),
    (axes[1], 'peak_MAPE', 'peak_MAPE', 'Peak-MAPE (%)', 'Peak-Weighted MAPE (top 10%)'),
]:
    use_peak = (peak_key is not None)
    result_pool = tx_results  # contains both base and +Tx

    base_d, aug_d = {}, {}
    for r in result_pool:
        m = re.match(r'(LGBM) (base|\+Tx) h\+(\d+)', r['name'])
        if m:
            tag = m.group(2)
            h   = int(m.group(3))
            key = 'peak_MAPE' if use_peak and 'peak_MAPE' in r else 'MAPE'
            val = r.get(key)
            if val is not None:
                if tag == 'base': base_d[h] = val
                else:             aug_d[h]  = val

    hs = sorted(base_d)
    ax.plot(hs, [base_d[h] for h in hs], label='Without Tx lead',
            color='#607D8B', marker='o', linewidth=2, markersize=7)
    ax.plot(hs, [aug_d[h]  for h in hs], label='With oracle Tx lead',
            color='#4CAF50', marker='s', linewidth=2, markersize=7, linestyle='--')

    ax.set_xlabel('Horizon (hours ahead)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(HORIZONS)
    ax.legend()
    ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('lgbm_tx_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: lgbm_tx_comparison.png")

In [ ]:
# ---------------------------------------------------------------------------
# Figure 4: Residual analysis for h+3 (middle horizon) — LightGBM
# Reuse evaluate.py helpers from notebook 01
# ---------------------------------------------------------------------------

h_inspect = 3
X_te, y_te = test_cache[h_inspect]
lgbm_inspect = models_cache[('LGBM', h_inspect)]
preds = lgbm_inspect.predict(X_te)

plot_residuals(y_te, preds, title=f'LGBM h+{h_inspect} — Residuals')
plot_residuals_by_time(y_te, preds, title=f'LGBM h+{h_inspect} — Residuals by Hour of Day')

In [ ]:
# ---------------------------------------------------------------------------
# Feature importance comparison across horizons (RF)
# Shows how feature rankings shift as horizon grows
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Random Forest Feature Importance by Horizon', fontsize=14)

for ax, h in zip(axes.flat, HORIZONS):
    rf_model  = models_cache[('RF', h)]
    feat_list = features_cache[h]
    imp_df    = get_feature_importance(rf_model, feat_list).head(10)

    ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color='#2196F3', alpha=0.8)
    ax.set_title(f'h+{h}')
    ax.set_xlabel('Importance')
    ax.tick_params(labelsize=8)
    ax.grid(axis='x', alpha=0.35)

plt.tight_layout()
plt.savefig('feature_importance_by_horizon.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_importance_by_horizon.png")

In [ ]:
# ---------------------------------------------------------------------------
# Final printed summary table
# ---------------------------------------------------------------------------

summary_rows = []
for h in HORIZONS:
    for model in ('RF', 'LGBM'):
        r_glob = next(r for r in horizon_results      if r['name'] == f'{model} h+{h}')
        r_peak = next(r for r in horizon_peak_results if r['name'] == f'{model} h+{h}')
        summary_rows.append({
            'Horizon' : f'h+{h}',
            'Model'   : model,
            'MAPE (%)'     : round(r_glob['MAPE'],      3),
            'RMSE'         : round(r_glob['RMSE'],      1),
            'Peak-MAPE (%)': round(r_peak['peak_MAPE'], 3),
        })

summary_df = pd.DataFrame(summary_rows)
print("\n=== Final Summary ===")
print(summary_df.to_string(index=False))

summary_df.to_csv('notebook02_summary.csv', index=False)
print("\nSaved: notebook02_summary.csv")